### RAG Pipelines- Data Ingestion to Vector Database Pipeline 

In [45]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [46]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""

    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively in the directory
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing file: {pdf_file}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)

            print(f"Loaded {len(documents)} pages")

        except Exception as e:
            print(f"Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")

    return all_documents

In [47]:
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data") 

Found 6 PDF files to process.

Processing file: ..\data\pdf\A better way to teach children to read in India.pdf
Loaded 42 pages

Processing file: ..\data\pdf\Attention.pdf
Loaded 44 pages

Processing file: ..\data\pdf\EC4102_MidTerm_Proposal.pdf
Loaded 5 pages

Processing file: ..\data\pdf\Mid_term_applied_econometrics_group_6.pdf
Loaded 5 pages

Processing file: ..\data\pdf\SadakVision — INDIS 2026.pdf
Loaded 16 pages

Processing file: ..\data\pdf\SV_IEEE.pdf
Loaded 8 pages

Total documents loaded: 120


In [48]:
all_pdf_documents

[Document(metadata={'producer': 'Acrobat Distiller 9.2.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2010-01-19T12:19:10-05:00', 'author': 'Leigh', 'moddate': '2010-01-19T12:36:55-05:00', 'title': 'Microsoft Word - Shishu Working Paper 2009-06-1', 'source': '..\\data\\pdf\\A better way to teach children to read in India.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'source_file': 'A better way to teach children to read in India.pdf', 'file_type': 'pdf'}, page_content="A Better Way to Teach Children to Read? \nEvidence from a Randomized Controlled Trial1 \n \n \n \n \nMay 2009 \n \nFang He \n(Columbia University) \n \nLeigh L. Linden2 \n(Columbia University, BREAD, and IZA) \n \nMargaret MacLeod \n(Columbia University and U.S. Department of State) \n \n \n \n \n \n \n \n \nAbstract: Using\xa0three\xa0randomized\xa0evaluations,\xa0we\xa0assess\xa0the\xa0relative\xa0efficacy\xa0of\xa0a\xa0literacy\xa0skills\xa0\ndevelopment\xa0program\xa0in\xa0Mumbai,\xa

### Text Splitting and making Chunks

In [49]:
def split_documents(documents, chunk_size = 1000, chunk_overlap = 200):
    """Split documents into smaller chunks"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    #Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata} ")
    return split_docs
   

In [50]:
chunks = split_documents(all_pdf_documents)

Split 120 documents into 320 chunks

Example chunk:
Content: A Better Way to Teach Children to Read? 
Evidence from a Randomized Controlled Trial1 
 
 
 
 
May 2009 
 
Fang He 
(Columbia University) 
 
Leigh L. Linden2 
(Columbia University, BREAD, and IZA) 
 
...
Metadata: {'producer': 'Acrobat Distiller 9.2.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2010-01-19T12:19:10-05:00', 'author': 'Leigh', 'moddate': '2010-01-19T12:36:55-05:00', 'title': 'Microsoft Word - Shishu Working Paper 2009-06-1', 'source': '..\\data\\pdf\\A better way to teach children to read in India.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'source_file': 'A better way to teach children to read in India.pdf', 'file_type': 'pdf'} 


### Embedding and Vector Store DB

In [51]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [52]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(
        self,
        model_name: str = "all-MiniLM-L6-v2",
    ):
        """Initialize the EmbeddingManager."""
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self) -> None:
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully: {self.model_name}. "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading embedding model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts."""
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True,
            convert_to_numpy=True,
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

In [53]:
# initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1531.27it/s]


Model loaded successfully: all-MiniLM-L6-v2. Embedding dimension: 384


C:\Users\Admin\AppData\Local\Temp\ipykernel_11800\3387099033.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


### VectorStore

In [54]:
class VectorStore:
    """Handles storage of document embeddings using ChromaDB."""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persistent_directory: str = "../data/vector_store"
    ):
        """
        Initialize the VectorStore.

        Args:
            collection_name (str): Name of the ChromaDB collection.
            persistent_directory (str): Directory where the vector
                store will be persisted.
        """

        self.collection_name = collection_name
        self.persistent_directory = persistent_directory

        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self) -> None:
        """Initialize the persistent ChromaDB vector store."""

        try:
            # Create the directory if it doesn't exist
            os.makedirs(
                self.persistent_directory,
                exist_ok=True
            )

            # Create persistent ChromaDB client
            self.client = chromadb.PersistentClient(
                path=self.persistent_directory
            )

            # Get existing collection or create a new one
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )

            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(
        self,
        documents: List[Any],
        embeddings: np.ndarray
    ) -> None:
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain Document objects.
            embeddings: Corresponding embeddings for the documents.
        """

        if not documents or embeddings.size == 0:
            raise ValueError(
                "Documents and embeddings cannot be empty"
            )

        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents and embeddings must match"
            )

        print(
            f"Adding {len(documents)} documents "
            f"to the vector store..."
        )

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(
            zip(documents, embeddings)
        ):

            # Generate unique document ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)

            metadata["doc_index"] = i
            metadata["content_length"] = len(
                doc.page_content
            )

            metadatas.append(metadata)

            # Store original document text
            documents_text.append(
                doc.page_content
            )

            # Convert NumPy array to Python list
            embeddings_list.append(
                embedding.tolist()
            )

        # Add everything to ChromaDB
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} "
                f"documents to the vector store."
            )

            print(
                f"Total documents in collection after addition: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(
                f"Error adding documents to vector store: {e}"
            )
            raise

In [55]:
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 1515


In [56]:
chunks

[Document(metadata={'producer': 'Acrobat Distiller 9.2.0 (Windows)', 'creator': 'PScript5.dll Version 5.2.2', 'creationdate': '2010-01-19T12:19:10-05:00', 'author': 'Leigh', 'moddate': '2010-01-19T12:36:55-05:00', 'title': 'Microsoft Word - Shishu Working Paper 2009-06-1', 'source': '..\\data\\pdf\\A better way to teach children to read in India.pdf', 'total_pages': 42, 'page': 0, 'page_label': '1', 'source_file': 'A better way to teach children to read in India.pdf', 'file_type': 'pdf'}, page_content='A Better Way to Teach Children to Read? \nEvidence from a Randomized Controlled Trial1 \n \n \n \n \nMay 2009 \n \nFang He \n(Columbia University) \n \nLeigh L. Linden2 \n(Columbia University, BREAD, and IZA) \n \nMargaret MacLeod \n(Columbia University and U.S. Department of State) \n \n \n \n \n \n \n \n \nAbstract: Using\xa0three\xa0randomized\xa0evaluations,\xa0we\xa0assess\xa0the\xa0relative\xa0efficacy\xa0of\xa0a\xa0literacy\xa0skills\xa0\ndevelopment\xa0program\xa0in\xa0Mumbai,\xa

In [57]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings for the document chunks
embeddings = embedding_manager.generate_embeddings(texts)

###Store in the vector store
vector_store.add_documents(chunks, embeddings)

Generating embeddings for 320 texts...


Batches: 100%|██████████| 10/10 [00:15<00:00,  1.58s/it]


Generated embeddings with shape: (320, 384)
Adding 320 documents to the vector store...
Successfully added 320 documents to the vector store.
Total documents in collection after addition: 1835


### RAG Retriever from VectorStore (use query to retrieve relevant documents)

In [58]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        """
        Initialize the retriever.

        Args:
            vector_store: Vector store containing document embeddings.
            embedding_manager: Manager for generating query embeddings.
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.

        Args:
            query: The search query.
            top_k: Number of top results to return.
            score_threshold: Minimum similarity score required.

        Returns:
            List of dictionaries containing:
            - document ID
            - document content
            - metadata
            - similarity score
            - distance
            - rank
        """

        print(f"Retrieving documents for query: '{query}'")
        print(
            f"Top K: {top_k}, "
            f"Score threshold: {score_threshold}"
        )

        # --------------------------------------------------
        # 1. Generate embedding for the user's query
        # --------------------------------------------------

        query_embedding = (
            self.embedding_manager
            .generate_embeddings([query])[0]
        )

        # --------------------------------------------------
        # 2. Search the vector store
        # --------------------------------------------------

        try:

            results = self.vector_store.collection.query(
                query_embeddings=[
                    query_embedding.tolist()
                ],
                n_results=top_k
            )

            # --------------------------------------------------
            # 3. Process the search results
            # --------------------------------------------------

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:

                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                # --------------------------------------------------
                # 4. Process each retrieved document
                # --------------------------------------------------

                for i, (
                    doc_id,
                    document,
                    metadata,
                    distance
                ) in enumerate(
                    zip(
                        ids,
                        documents,
                        metadatas,
                        distances
                    )
                ):

                    # ChromaDB returns cosine distance.
                    # Convert distance to similarity score.
                    similarity_score = 1 - distance

                    # --------------------------------------------------
                    # 5. Apply similarity threshold
                    # --------------------------------------------------

                    if similarity_score >= score_threshold:

                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(
                    f"Retrieved {len(retrieved_docs)} "
                    f"documents (after filtering)"
                )

            else:

                print("No documents found")

            return retrieved_docs

        except Exception as e:

            print(f"Error during retrieval: {e}")

            return []

In [59]:
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

In [60]:
rag_retriever.retrieve("What is sadak vision?")

Retrieving documents for query: 'What is sadak vision?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 43.60it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_1effd827_176',
  'content': 'Vision-level accuracy, 89% less compute , plus severity\nSensor only Vision only Fusion (SadakVision)\n0.48\n0.782 0.782 0.823 0.787 0.787\n0.606\n0.785 0.785\nPrecision Recall F1\nSame F1\nas vision-only, by design: the sensor decides what\nthe camera looks at, vision gives the verdict.\n 89%\nframes sent to the detector: 4,620 vs 43,200\nan hour.\n+ Severity\nlow / medium / high from the sensor trace.\nChecked so far on 62 human-labelled events.\nSadakVision · INDIS 2026 10',
  'metadata': {'source': '..\\data\\pdf\\SadakVision — INDIS 2026.pdf',
   'doc_index': 176,
   'page_label': '10',
   'page': 9,
   'total_pages': 16,
   'source_file': 'SadakVision — INDIS 2026.pdf',
   'file_type': 'pdf',
   'producer': 'appifact kit',
   'creationdate': '',
   'creator': 'PyPDF',
   'title': 'SadakVision   INDIS 2026',
   'content_length': 464},
  'similarity_score': 0.09583026170730591,
  'distance': 0.9041697382926941,
  'rank': 1},
 {'id': 'doc_7b

### Intergration VectorDB Context pipeline with LLM output

In [61]:
### Simple RAG pipeline with Groq LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv


# Load environment variables from .env
load_dotenv()


# Get Groq API key
groq_api_key = os.getenv("GROQ_API_KEY")

if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file")


# Initialize the Groq LLM
llm = ChatGroq(
    groq_api_key=groq_api_key,
    # model_name="gemma2-9b-it",
    model_name = "openai/gpt-oss-20b",
    temperature=0.1,
    max_tokens=1024
)


# Simple RAG pipeline:
# Retrieve relevant context + generate answer
def rag_simple(query, retriever, llm, top_k=3):

    # 1. Retrieve relevant documents
    results = retriever.retrieve(
        query,
        top_k=top_k
    )

    # 2. Combine retrieved document chunks into one context
    context = "\n\n".join(
        doc["content"]
        for doc in results
    ) if results else ""

    # 3. Handle case where no relevant documents are found
    if not context:
        print("No relevant documents found for the query.")
        return "No relevant documents found."

    # 4. Create the prompt for the LLM
    prompt = f"""
Use the following context to answer the question concisely.
Only use information from the provided context.

Context:
{context}

Question:
{query}

Answer:
"""

    # 5. Send the prompt to the Groq LLM
    response = llm.invoke(prompt)

    # 6. Return the generated answer
    return response.content




In [62]:
# Example usage
query = "What is the Attention Mechanism?"

answer = rag_simple(
    query=query,
    retriever=rag_retriever,
    llm=llm,
    top_k=3
)

print("\nFinal Answer:")
print(answer)

Retrieving documents for query: 'What is the Attention Mechanism?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 41.20it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)



Final Answer:
The attention mechanism is a way that individual attention heads in a model learn to focus on particular positions or patterns in the input. For example, some heads attend mainly to the next or previous token, others follow syntactic relations such as verbs to subjects or pronouns to antecedents, and some broadcast attention to special tokens that receive input from many positions.


### Enhanced RAG Pipeline Features

In [63]:
# --- Enhanced RAG Pipeline Features ---

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])

    sources = [{
        'source': doc['metadata'].get(
            'source_file',
            doc['metadata'].get('source', 'unknown')
        ),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:120] + '...'
    } for doc in results]

    confidence = max(
        [doc['similarity_score'] for doc in results]
    )

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke(
        [prompt.format(context=context, query=query)]
    )

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context

    return output

In [64]:
# Example usage of the advanced RAG pipeline
result = rag_advanced(
    query="what is better way to teach attention mechanism  ?",
    retriever=rag_retriever,
    llm=llm,
    top_k=5,
    min_score=0.005,
    return_context=True
)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence Score:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is better way to teach attention mechanism  ?'
Top K: 5, Score threshold: 0.005
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 62.67it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


Answer: A concise, effective way to teach the attention mechanism is to combine **intuition, visualisation, and hands‑on coding** in a progressive sequence:

1. **Start with a simple analogy** (e.g., a spotlight focusing on relevant words in a sentence).  
2. **Show a visual diagram** of query, key, value vectors and the soft‑max weighting, letting students see how scores are computed and how the weighted sum is formed.  
3. **Walk through the math step‑by‑step**:  
   - Compute dot‑product scores.  
   - Apply temperature scaling and soft‑max.  
   - Multiply by values and sum.  
4. **Implement a minimal example in code** (Python + NumPy or PyTorch) that students can run, modify, and observe the effect of changing queries or keys.  
5. **Gradually introduce complexity**: multi‑head attention, positional encodings, and scaling to larger models.  
6. **End with a small project** (e.g., a toy translation or sentiment‑analysis task) where students train a tiny Transformer and analyse atte

In [65]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---

from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:

    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(
        self,
        question: str,
        top_k: int = 5,
        min_score: float = 0.2,
        stream: bool = False,
        summarize: bool = False
    ) -> Dict[str, Any]:

        # Retrieve relevant documents
        results = self.retriever.retrieve(
            question,
            top_k=top_k,
            score_threshold=min_score
        )

        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""

        else:
            context = "\n\n".join(
                [doc["content"] for doc in results]
            )

            sources = [
                {
                    "source": doc["metadata"].get(
                        "source_file",
                        doc["metadata"].get("source", "unknown")
                    ),
                    "page": doc["metadata"].get(
                        "page",
                        "unknown"
                    ),
                    "score": doc["similarity_score"],
                    "preview": doc["content"][:120] + "..."
                }
                for doc in results
            ]

            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {question}

Answer:"""

            if stream:
                print("Streaming answer:")

                for i in range(0, len(prompt), 80):
                    print(
                        prompt[i:i + 80],
                        end="",
                        flush=True
                    )
                    time.sleep(0.05)

                print()

            response = self.llm.invoke(
                [prompt.format(
                    context=context,
                    question=question
                )]
            )

            answer = response.content

        # Add citations to answer
        citations = [
            f"[{i + 1}] {src['source']} (page {src['page']})"
            for i, src in enumerate(sources)
        ]

        answer_with_citations = (
            answer
            + "\n\nCitations:\n"
            + "\n".join(citations)
            if citations
            else answer
        )

        # Optionally summarize answer
        summary = None

        if summarize and answer:
            summary_prompt = (
                f"Summarize the following answer in 2 sentences:\n"
                f"{answer}"
            )

            summary_resp = self.llm.invoke(
                [summary_prompt]
            )

            summary = summary_resp.content

        # Store query history
        self.history.append({
            "question": question,
            "answer": answer,
            "sources": sources,
            "summary": summary
        })

        return {
            "question": question,
            "answer": answer_with_citations,
            "sources": sources,
            "summary": summary,
            "history": self.history
        }


In [66]:
# Example usage
adv_rag = AdvancedRAGPipeline(
    rag_retriever,
    llm
)

result = adv_rag.query(
    "what is attention mechanism",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)

print("\nFinal Answer:", result["answer"])
print("Summary:", result["summary"])
print("History:", result["history"][-1])

Retrieving documents for query: 'what is attention mechanism'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 44.62it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
(see Table A.4 for a complete reference).
Definition 2.1(Attention Mechanism).An attention mechanism is a differentiable functionA:R dq×
Rn×dk×Rn×dv→R dv that computes a weighted combination of values based on the compatibility between
a query and key

s.
Given:
•Query:q∈R dq
•Keys:K= (k 1,...,k n) wherek i∈R dk
•Values:V= (v 1,...,v n) wherev i∈R dv
The attention function computes:
ei = score(q,ki) fori= 1,...,n(2.8)
αi = exp(ei)Pn
j=1 exp(ej) (2.9)
A(q,K,V) =
nX
i=1
αivi (2.10)
where score :Rdq×Rdk→Ris a compatibility function, andα i are attention weights satisfyingPn
i=1αi =
1 andα i≥0.
The attention mechanism consists of three conceptual stages:
Stage 1 - Score Computation: For each positioni, compute a scalar scoree i measuring the com-
patibility or relevance between the queryqand keyk i. Different scoring functions are explored in the next
section.

(see Table A.4 for a complete reference).
Definition 2.1(Attention Mechanism).An attention mechanism is a differentiable functionA:R dq×
Rn×dk×Rn×dv→R dv that computes a weighted combination of values based on the compatibility between
a query and keys.
Given:
•Query:q∈R dq
•Keys:K= (k 1,...,k n) wherek i∈R dk
•Values:V= (v 1,...,v n) wherev i∈R dv
The attention function computes: